# Image Region Exploration

Explore classical-CV region detection with a generated scanned-page fixture. Set `SAMPLE_IMAGE_PATH` to inspect a real page. Outputs are written under `data/notebook_outputs/`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import cv2
import matplotlib.pyplot as plt
import numpy as np

from utils.image.layout.analyzer import LayoutAnalyzer
from utils.image.ops.extract import save_region_crops
from martial_arts_ocr.pipeline.extraction_adapters import image_region_from_extraction

OUTPUT_DIR = PROJECT_ROOT / "data" / "notebook_outputs" / "01_image_regions"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_IMAGE_PATH = None  # Example: PROJECT_ROOT / "data" / "sample.png"

In [ ]:
def synthetic_page():
    page = np.full((520, 420, 3), 255, dtype=np.uint8)
    for y in range(42, 210, 20):
        cv2.line(page, (42, y), (250, y), (0, 0, 0), 2)
    cv2.rectangle(page, (70, 270), (245, 395), (235, 235, 235), -1)
    cv2.rectangle(page, (70, 270), (245, 395), (0, 0, 0), 3)
    cv2.line(page, (92, 367), (220, 298), (0, 0, 0), 3)
    cv2.circle(page, (190, 338), 22, (0, 0, 0), 3)
    cv2.rectangle(page, (350, 455), (354, 459), (0, 0, 0), -1)  # tiny noise
    return page

if SAMPLE_IMAGE_PATH:
    image = cv2.imread(str(SAMPLE_IMAGE_PATH))
else:
    image = synthetic_page()

plt.figure(figsize=(6, 8))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.axis("off");

In [ ]:
analyzer = LayoutAnalyzer({
    "use_yolo_figure": False,
    "enabled_detectors": ["figure", "contours"],
    "contours_always": True,
    "figure_min_area": 2500,
    "contour_min_area": 2500,
    "filter_text_like": True,
})
regions = analyzer.detect_image_regions(image)
print(f"Detected {len(regions)} image-like regions")
for region in regions:
    print(region.to_dict())

In [ ]:
overlay = analyzer.debug_draw_regions(image, regions)
plt.figure(figsize=(6, 8))
plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
plt.axis("off");

In [ ]:
crop_records = save_region_crops(image, regions, OUTPUT_DIR, prefix="diagram")
canonical_regions = [image_region_from_extraction(item, index=i) for i, item in enumerate(crop_records, start=1)]
print(crop_records)
print([region.to_dict() for region in canonical_regions])